# S&P 500 Options: NLinear

This notebook snapshots the complete three-model sequence population before fitting its NLinear
member. `09a_lstm` and `09b_patchtst` execute the other declared members against the same
immutable population. Every configured checkpoint remains eligible for model analysis and
backtesting.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit NLinear within the declared S&P 500 options sequence population."""

import polars as pl

from case_studies.sp500_options.research_workflow import (
    ALL_LABELS,
    declared_dl_device,
    model_request_catalog,
    open_study,
    published_dl_device,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_subset,
    run_resolved_model_requests,
    snapshot_official_model_catalog,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
DEVICE: str = ""

SEQUENCE_CONFIGS = ("nlinear", "lstm_h64", "patchtst")
POPULATION_NAME: str = ""
SUPERSEDES_POPULATION: str = "fd45f829a576"

### The device the population was fitted on

A network trained on a GPU and the same network trained on a CPU accumulate their sums in a
different order and reach different weights, so the device is part of what the fitted model is
and sits inside the training identity rather than beside it. The device this population was
fitted on is declared once, in `modeling.dl.device` in `config/setup.yaml`, and read from there
by all four deep-learning notebooks rather than retyped in each. On a machine with no NVIDIA
card the run stops here rather than quietly training something else: set `DEVICE="cpu"` and pass
a `POPULATION_NAME` to fit the same requests there, under a name of their own.

In [3]:
CANONICAL_POPULATION_NAME = "sp500-options-sequence-validation-v1"

published_device = published_dl_device()
device = declared_dl_device(DEVICE)
population_name = POPULATION_NAME or CANONICAL_POPULATION_NAME
if device != published_device and population_name == CANONICAL_POPULATION_NAME:
    raise ValueError(
        f"this run fits on {device!r}, not the published {published_device!r}, so its "
        f"identities are not the ones {CANONICAL_POPULATION_NAME!r} holds; pass "
        f"POPULATION_NAME to give them a population of their own"
    )
print(f"training device: {device} (declared: {published_device})")

training device: cuda (declared: cuda)


## Complete sequence request population

The case-wide table is resolved before the first member executes. Canonical execution snapshots
all configuration-checkpoint identities so a failed member cannot disappear from later analysis.

**A name holds one generation at a time**, and this notebook is the only one that writes this
population - `09a_lstm` and `09b_patchtst` execute members of a snapshot that already exists.
Anything that moves a training identity moves every prediction hash with it, so the members
this run computes are no longer the members an earlier snapshot under the same name declared,
and those two notebooks then refuse their own work as undeclared. `SUPERSEDES_POPULATION`
names the snapshot such a run retires, and the value is part of what the population is hashed
over. The value here names the snapshot this run retires; it is empty only for the first
snapshot under a name.

`create` refuses a changed member list under an existing name unless this names the current
snapshot, so the parameter is what makes refreshing this population possible at all. Without
it the refit stops at the write with the hash it needs, which is the right failure but not
one this notebook could act on.

In [4]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
all_requests = model_request_catalog(
    "deep_learning",
    labels=ALL_LABELS,
    config_names=SEQUENCE_CONFIGS,
)
all_resolved = resolve_model_requests(
    study,
    all_requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_model_plan(all_resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""regression""",52,248,42006,2,2019-01-07 00:00:00,2020-11-10 00:00:00,20,"""canonical""","""6038372bdd4d"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""regression""",52,248,42006,2,2019-01-07 00:00:00,2020-11-10 00:00:00,20,"""canonical""","""dc76d91d861b"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""regression""",52,248,42006,2,2019-01-07 00:00:00,2020-11-10 00:00:00,20,"""canonical""","""472344b774ad"""


## Execute NLinear

NLinear shares the gap-safe sequence construction, fold boundaries, fitted-state persistence,
restart, and exact eligible-key checks used by the other sequence configurations.

In [5]:
nlinear_resolved = tuple(
    request for request in all_resolved if request.spec["config_name"] == "nlinear"
)
if len(nlinear_resolved) != 1:
    raise ValueError("the sequence population must contain exactly one NLinear request")

if EXECUTION_TIER == "canonical":
    population = snapshot_official_model_catalog(
        study,
        all_requests,
        population_name=population_name,
        resolved_requests=all_resolved,
        supersedes=SUPERSEDES_POPULATION or None,
    )
    execution, population = run_official_model_subset(
        study,
        nlinear_resolved,
        population=population,
    )
else:
    if not WORKSPACE or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, nlinear_resolved)
    population = None

Fold-major CV: 2 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=49,609 seq across 472 symbols
    val=12,146 seq across 473 symbols
    creating datasets...
    datasets ready
    nlinear:


      epoch   1/100: train_loss=0.817201


      epoch   2/100: train_loss=0.663327


      epoch   3/100: train_loss=0.626412


...

      epoch  99/100: train_loss=0.635781


      epoch 100/100: train_loss=0.634989, val_loss=0.595146, IC=-0.0143


      best_ep=5, IC=-0.0010 (44.1s, 20 checkpoints)


  nlinear: best_epoch=5, IC=-0.0062 (100.5s)



  Best: nlinear @ epoch 5 (IC=-0.0062)
  Saved to ~/ml4t/public-s6-sp500_options/case_studies/sp500_options/run_log/training/dc76d91d861b/diagnostics


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("NLinear execution returned a partial checkpoint")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",5,"""canonical""",true,"""dc76d91d861b""","""0f69f89a70ae"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",10,"""canonical""",true,"""dc76d91d861b""","""3b7f25dc193c"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",15,"""canonical""",true,"""dc76d91d861b""","""585bcf2117e4"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",20,"""canonical""",true,"""dc76d91d861b""","""0372c9f2f678"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",25,"""canonical""",true,"""dc76d91d861b""","""b845bbf18515"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",80,"""canonical""",true,"""dc76d91d861b""","""de22cc6d47c6"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",85,"""canonical""",true,"""dc76d91d861b""","""d1102ee90772"""
"""deep_learning""","""ret_to_expiry""","""nlinear""","""epoch""",90,"""canonical""",true,"""dc76d91d861b""","""a1177fb06793"""


The NLinear checkpoint artifacts are complete. The official sequence population remains open
until `09a_lstm` and `09b_patchtst` publish their declared members.